# DoodleVPN: почему доход падает при росте онлайна

Срез на **2026-07-30**, бизнес-период начинается строго **2026-04-13**. Ноутбук содержит только агрегированные данные и воспроизводит ключевые расчёты отчёта.

**Основные источники**

- `/opt/accounting/data/accounting.db`: `revenue`, `revenue_refunds` — доход после комиссии и возвратов.
- `/opt/accounting/data/bot_mirror.db`: `users`, `platega_payments`, `crypto_payments`, `payment_url_clicks`, `start_attributions`, `broadcasts` — регистрации, trial, платёжная воронка и атрибуция.
- Remnawave PostgreSQL: `public.nodes_user_usage_history`, `public.users` — DAU/MAU и трафик.

Все даты текущего месяца частичные: июль — по 30 июля. История Remnawave usage начинается 17 апреля, поэтому сравнение онлайна ведётся с мая.

In [1]:
import math
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

## 1. Месячная экономика и онлайн

`net_revenue_usd` = сумма `revenue.amount_usd_net` минус связанные возвраты. Разложение new/returning использует доход после комиссии до возвратов (возвраты за весь период — только $18.77 и не меняют вывод).

In [2]:
monthly = pd.DataFrame([
    {"month":"2026-05", "net_revenue_usd":3581.06, "transactions":730, "payers":644, "aov_usd":4.91,
     "registrations":892, "new_payers":525, "new_revenue_usd":3009.63, "returning_payers":119, "returning_revenue_usd":574.73,
     "all_mau":1564, "tg_mau":1552, "restored_mau":4, "all_dau":823.9, "tg_dau":820.7, "restored_dau":0.3,
     "traffic_gb":91314.1, "tg_traffic_gb":87091.0, "restored_traffic_gb":7.2},
    {"month":"2026-06", "net_revenue_usd":3066.94, "transactions":642, "payers":576, "aov_usd":4.78,
     "registrations":627, "new_payers":257, "new_revenue_usd":1662.92, "returning_payers":319, "returning_revenue_usd":1404.02,
     "all_mau":1674, "tg_mau":1470, "restored_mau":201, "all_dau":992.1, "tg_dau":966.6, "restored_dau":24.6,
     "traffic_gb":134404.6, "tg_traffic_gb":133146.4, "restored_traffic_gb":1175.1},
    {"month":"2026-07", "net_revenue_usd":2816.42, "transactions":566, "payers":513, "aov_usd":4.98,
     "registrations":419, "new_payers":210, "new_revenue_usd":1467.50, "returning_payers":303, "returning_revenue_usd":1364.39,
     "all_mau":1921, "tg_mau":1450, "restored_mau":468, "all_dau":1150.5, "tg_dau":1061.8, "restored_dau":87.5,
     "traffic_gb":181392.0, "tg_traffic_gb":174358.4, "restored_traffic_gb":5788.9},
])
monthly["tx_per_payer"] = monthly.transactions / monthly.payers
monthly["revenue_per_active_day"] = monthly.net_revenue_usd / (monthly.all_dau * [31, 30, 30])
monthly["revenue_per_gb"] = monthly.net_revenue_usd / monthly.traffic_gb
monthly

     month  net_revenue_usd  transactions  payers  aov_usd  registrations  \
0  2026-05         3,581.06           730     644     4.91            892   
1  2026-06         3,066.94           642     576     4.78            627   
2  2026-07         2,816.42           566     513     4.98            419   

   new_payers  new_revenue_usd  returning_payers  returning_revenue_usd  \
0         525         3,009.63               119                 574.73   
1         257         1,662.92               319               1,404.02   
2         210         1,467.50               303               1,364.39   

   all_mau  tg_mau  restored_mau  all_dau   tg_dau  restored_dau  traffic_gb  \
0     1564    1552             4   823.90   820.70          0.30   91,314.10   
1     1674    1470           201   992.10   966.60         24.60  134,404.60   
2     1921    1450           468 1,150.50 1,061.80         87.50  181,392.00   

   tg_traffic_gb  restored_traffic_gb  tx_per_payer  revenue_per_acti

In [3]:
def pct_change(a, b):
    return 100 * (b / a - 1)

may, jun, jul = (monthly.iloc[i] for i in range(3))
headline = pd.Series({
    "revenue May→Jul": pct_change(may.net_revenue_usd, jul.net_revenue_usd),
    "payers May→Jul": pct_change(may.payers, jul.payers),
    "AOV May→Jul": pct_change(may.aov_usd, jul.aov_usd),
    "all MAU May→Jul": pct_change(may.all_mau, jul.all_mau),
    "core tg MAU May→Jul": pct_change(may.tg_mau, jul.tg_mau),
    "core tg DAU May→Jul": pct_change(may.tg_dau, jul.tg_dau),
    "core tg traffic May→Jul": pct_change(may.tg_traffic_gb, jul.tg_traffic_gb),
    "revenue / active-day May→Jul": pct_change(may.revenue_per_active_day, jul.revenue_per_active_day),
    "revenue / GB May→Jul": pct_change(may.revenue_per_gb, jul.revenue_per_gb),
}).round(1)
headline.to_frame("change_pct")

                              change_pct
revenue May→Jul                   -21.40
payers May→Jul                    -20.30
AOV May→Jul                         1.40
all MAU May→Jul                    22.80
core tg MAU May→Jul                -6.60
core tg DAU May→Jul                29.40
core tg traffic May→Jul           100.20
revenue / active-day May→Jul      -41.80
revenue / GB May→Jul              -60.40

### Разложение падения дохода

Тождество: `revenue = unique payers × transactions per payer × average transaction value`. Последовательное разложение сохраняет точную сумму изменения.

In [4]:
def decompose(start, end):
    p0, f0, a0 = start.payers, start.tx_per_payer, start.net_revenue_usd / start.transactions
    p1, f1, a1 = end.payers, end.tx_per_payer, end.net_revenue_usd / end.transactions
    return pd.Series({
        "payer_count_effect": (p1 - p0) * f0 * a0,
        "frequency_effect": p1 * (f1 - f0) * a0,
        "aov_effect": p1 * f1 * (a1 - a0),
        "total_change": end.net_revenue_usd - start.net_revenue_usd,
    }).round(2)

decomposition = pd.DataFrame({
    "May→Jun": decompose(may, jun),
    "Jun→Jul": decompose(jun, jul),
    "May→Jul": decompose(may, jul),
})
decomposition

                    May→Jun  Jun→Jul  May→Jul
payer_count_effect  -378.12  -335.45  -728.45
frequency_effect     -53.57   -27.62   -76.07
aov_effect           -82.43   112.54    39.87
total_change        -514.12  -250.52  -764.64

## 2. Почему «онлайн растёт» — неправильная бизнес-интерпретация

В Remnawave появился растущий технический сегмент `wl_restor*`: 4 MAU в мае, 201 в июне, 468 в июле. Ядро `tg_*` по MAU за это время снизилось. При этом DAU ядра и трафик выросли: меньше уникальных Telegram-аккаунтов используют сервис интенсивнее и/или имеют уже оплаченные длинные подписки. Онлайн — метрика потребления накопленного subscriber stock, cash revenue — поток новых оплат за месяц.

In [5]:
online_bridge = pd.DataFrame([
    {"metric":"Total MAU", "May":1564, "June":1674, "July":1921},
    {"metric":"Core tg_* MAU", "May":1552, "June":1470, "July":1450},
    {"metric":"Restored wl_restor* MAU", "May":4, "June":201, "July":468},
    {"metric":"Core tg_* DAU", "May":820.7, "June":966.6, "July":1061.8},
    {"metric":"Core tg_* traffic, GB", "May":87091.0, "June":133146.4, "July":174358.4},
])
online_bridge

                    metric       May       June       July
0                Total MAU  1,564.00   1,674.00   1,921.00
1            Core tg_* MAU  1,552.00   1,470.00   1,450.00
2  Restored wl_restor* MAU      4.00     201.00     468.00
3            Core tg_* DAU    820.70     966.60   1,061.80
4    Core tg_* traffic, GB 87,091.00 133,146.40 174,358.40

## 3. Воронка новых пользователей

Для честного сравнения июнь и июль ограничены одинаковыми окнами **1–23 число**; оплата считается в течение 7 дней после регистрации.

In [6]:
funnel = pd.DataFrame([
    {"cohort":"2026-06-01..23", "registered":525, "trial_started":377, "trial_rate_pct":71.8, "paid_7d":116, "paid_7d_pct":22.1, "trial_to_paid_7d_pct":22.5},
    {"cohort":"2026-07-01..23", "registered":304, "trial_started":199, "trial_rate_pct":65.5, "paid_7d":62, "paid_7d_pct":20.4, "trial_to_paid_7d_pct":20.6},
])
funnel

           cohort  registered  trial_started  trial_rate_pct  paid_7d  \
0  2026-06-01..23         525            377           71.80      116   
1  2026-07-01..23         304            199           65.50       62   

   paid_7d_pct  trial_to_paid_7d_pct  
0        22.10                 22.50  
1        20.40                 20.60  

In [7]:
jun_f, jul_f = funnel.iloc[0], funnel.iloc[1]
expected_july_at_june_volume = jun_f.registered * jul_f.paid_7d_pct / 100
volume_loss = expected_july_at_june_volume - jul_f.paid_7d
conversion_loss = jun_f.registered * (jun_f.paid_7d_pct - jul_f.paid_7d_pct) / 100
pd.Series({
    "registrations_change_pct": pct_change(jun_f.registered, jul_f.registered),
    "paid_7d_change_pct": pct_change(jun_f.paid_7d, jul_f.paid_7d),
    "conversion_change_pp": jul_f.paid_7d_pct - jun_f.paid_7d_pct,
    "estimated_share_of_paid7_decline_from_volume_pct": 100 * volume_loss / (volume_loss + conversion_loss),
}).round(1).to_frame("value")

                                                  value
registrations_change_pct                         -42.10
paid_7d_change_pct                               -46.60
conversion_change_pp                              -1.70
estimated_share_of_paid7_decline_from_volume_pct  83.50

## 4. Тарифы, удержание и платёжные провайдеры

- Возвратный доход почти не изменился между июнем и июлем: $1,404 → $1,364.
- 1-месячное продление в окне 21–45 дней снизилось умеренно: 65.2% → 59.8% для сопоставимых когорт первой половины мая/июня.
- Wata не выглядит текущим узким местом: clicked→paid вырос с 77.8% до 82.7%.
- Крипто-воронка требует отдельной проверки: settled/attempt упал с 56.0% в мае до ~32% в июне-июле; среди тех, кто нажал платёжную ссылку, clicked→paid стабилен около 57–60%, значит основной провал раньше или в неполной click-телеметрии.

In [8]:
plans = pd.DataFrame([
    ["2026-05","1m",387,1194.53],["2026-05","3m",148,1020.92],["2026-05","6m",60,716.10],["2026-05","1y",30,520.14],
    ["2026-06","1m",368,1223.92],["2026-06","3m",108,832.36],["2026-06","6m",38,471.77],["2026-06","1y",18,407.11],
    ["2026-07","1m",290,908.78],["2026-07","3m",140,1026.81],["2026-07","6m",53,635.28],["2026-07","1y",10,183.10],
], columns=["month","plan","transactions","net_revenue_usd"])
plans.pivot(index="plan", columns="month", values=["transactions","net_revenue_usd"])

      transactions                 net_revenue_usd                  
month      2026-05 2026-06 2026-07         2026-05  2026-06  2026-07
plan                                                                
1m          387.00  368.00  290.00        1,194.53 1,223.92   908.78
1y           30.00   18.00   10.00          520.14   407.11   183.10
3m          148.00  108.00  140.00        1,020.92   832.36 1,026.81
6m           60.00   38.00   53.00          716.10   471.77   635.28

In [9]:
payments = pd.DataFrame([
    ["Platega","2026-05",975,629,64.5,None],
    ["Platega","2026-06",593,325,54.8,None],
    ["Wata","2026-06",394,260,66.0,77.8],
    ["Wata","2026-07 through 23",576,393,68.2,82.7],
    ["Crypto","2026-05",141,79,56.0,None],
    ["Crypto","2026-06",112,36,32.1,56.8],
    ["Crypto","2026-07 through 23",67,21,31.3,60.0],
], columns=["provider","period","attempts","settled","settled_pct","clicked_to_paid_pct"])
payments

  provider              period  attempts  settled  settled_pct  \
0  Platega             2026-05       975      629        64.50   
1  Platega             2026-06       593      325        54.80   
2     Wata             2026-06       394      260        66.00   
3     Wata  2026-07 through 23       576      393        68.20   
4   Crypto             2026-05       141       79        56.00   
5   Crypto             2026-06       112       36        32.10   
6   Crypto  2026-07 through 23        67       21        31.30   

   clicked_to_paid_pct  
0                  NaN  
1                  NaN  
2                77.80  
3                82.70  
4                  NaN  
5                56.80  
6                60.00  

## 5. Контрольные проверки

Проверки специально падают, если ключевые агрегаты или интерпретации случайно меняются при редактировании ноутбука.

In [10]:
assert math.isclose(monthly.net_revenue_usd.sum(), 9464.42, abs_tol=0.01)
assert jul.net_revenue_usd < jun.net_revenue_usd < may.net_revenue_usd
assert jul.payers < jun.payers < may.payers
assert jul.aov_usd > may.aov_usd
assert jul.all_mau > may.all_mau and jul.tg_mau < may.tg_mau
assert payments.query("provider == 'Wata'").iloc[1].clicked_to_paid_pct > payments.query("provider == 'Wata'").iloc[0].clicked_to_paid_pct
assert funnel.iloc[1].paid_7d_pct >= 20
print("All checks passed")

All checks passed


## Вывод

Падение дохода — в первую очередь **провал объёма новых пользователей и новых плательщиков**, начавшийся примерно с недели 15 июня. Цена/AOV и основной Wata-флоу не объясняют падение. Рост «онлайна» смешивает технически восстановленные аккаунты с ядром и отражает растущую интенсивность уже оплаченного использования, поэтому не должен использоваться как прокси дохода.